In [3]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def calculate_rotation_angle_axis(R1, R2):
    """
    计算从旋转R1到R2所需绕世界坐标系旋转的轴和角度
    
    参数:
        R1: 世界系到物体系的旋转；初始旋转矩阵 (3x3) 或四元数 (4,) 或欧拉角 (3,)
        R2: 世界系到物体系的旋转；目标旋转矩阵 (3x3) 或四元数 (4,) 或欧拉角 (3,)
    
    返回:
        axis: 旋转轴 (世界坐标系下的单位向量)
        angle_deg: 旋转角度 (度数)
    """
    # 将输入转换为Rotation对象
    rot1 = R.from_matrix(R1) if R1.shape == (3,3) else R.from_quat(R1) if len(R1)==4 else R.from_euler('xyz', R1, degrees=True)
    rot2 = R.from_matrix(R2) if R2.shape == (3,3) else R.from_quat(R2) if len(R2)==4 else R.from_euler('xyz', R2, degrees=True)
    
    # 计算相对旋转: R2 = R_rel * R1 => R_rel = R2 * R1.inv()
    rot_rel = rot2 * rot1.inv()
    
    # 提取旋转轴和角度
    # as_rotvec() 返回的是旋转向量 (轴 * 角度(弧度))
    rotvec = rot_rel.as_rotvec()
    angle_rad = np.linalg.norm(rotvec)
    angle_deg = np.degrees(angle_rad)
    
    # 计算旋转轴（处理角度为0的特殊情况）
    if angle_rad < 1e-8:  # 角度接近0，无旋转
        axis = np.array([1, 0, 0])  # 默认返回x轴
        angle_deg = 0.0
    else:
        axis = rotvec / angle_rad  # 归一化得到旋转轴
    
    return axis, angle_deg

# ------------------- 测试示例 -------------------
if __name__ == "__main__":
    # 示例1: 初始旋转为单位矩阵（无旋转），目标旋转为绕z轴旋转90度
    R1 = np.eye(3)  # 单位旋转矩阵
    rot2 = R.from_euler('z', 90, degrees=True)
    R2 = rot2.as_matrix()
    
    axis, angle = calculate_rotation_angle_axis(R1, R2)
    print("示例1 - 旋转轴:", np.round(axis, 4))
    print("示例1 - 旋转角度(度):", np.round(angle, 4))
    print("-" * 50)
    
    # 示例2: 初始旋转为绕x轴旋转30度，目标旋转为绕y轴旋转60度
    rot1 = R.from_euler('x', 40, degrees=True)
    R1 = rot1.as_matrix()
    rot2 = R.from_euler('x', 60, degrees=True)
    R2 = rot2.as_matrix()
    
    axis, angle = calculate_rotation_angle_axis(R1, R2)
    print("示例2 - 旋转轴:", np.round(axis, 4))
    print("示例2 - 旋转角度(度):", np.round(angle, 4))




    #验证是否是绕世界坐标系定轴旋转

    rot1 = np.array([[0.0,0.0,-1.0],[0.0,1.0,0.0],[1.0,0.0,0.0]])
    rot2 = np.array([[0.0,-1.0,0.0],[0.0,0.0,-1.0],[1.0,0.0,0.0]])


    axis, angle = calculate_rotation_angle_axis(rot1, rot2)
    print("示例3 - 旋转轴:", np.round(axis, 4))
    print("示例3 - 旋转角度(度):", np.round(angle, 4))

示例1 - 旋转轴: [0. 0. 1.]
示例1 - 旋转角度(度): 90.0
--------------------------------------------------
示例2 - 旋转轴: [1. 0. 0.]
示例2 - 旋转角度(度): 20.0
示例3 - 旋转轴: [-0. -0.  1.]
示例3 - 旋转角度(度): 90.0


In [11]:

rotation_matrix_x_begin = np.array(
[
    [-0.977553005381213, 0.15400699859382183, 0.14377399518659545],
    [-0.003175601660714261, 0.6715560023482676, -0.7409469954501301],
    [-0.21066400779784725, -0.724772011110098, -0.6559930041831621],
])
rotation_matrix_x_end =  np.array(
[
    [0.08110289847201688, 0.6152490018997765, -0.7841499952245458],
    [0.9524440048690348, -0.2796869997478093, -0.1209349960285231],
    [-0.29372100815349755, -0.7370510112806542, -0.6086740047740324],
])


rotation_matrix_z_begin = np.array(
[
    [-0.9835740055868639, 0.0992885980936951, 0.15074099485132897],
    [-0.04269080182147626, 0.6834670023791757, -0.7287319956227514],
    [-0.17538100761517383, -0.7231970110286564, -0.6680020039622983],
])

rotation_matrix_z_end = np.array(
[
    [-0.26580200818734806, -0.6671960109856284, -0.6958440041300277],
    [-0.15461800195226083, 0.7419690029231333, -0.6523609960344168],
    [0.9515480046559502, -0.06580919869726254, -0.30037699465341117],
])



rotation_matrix_x_begin_for_compute = np.eye(3)
rotation_matrix_x_end_for_compute = rotation_matrix_x_end.T @ rotation_matrix_x_begin
rotation_matrix_z_begin_for_compute = np.eye(3)
rotation_matrix_z_end_for_compute = rotation_matrix_z_end.T @ rotation_matrix_z_begin


axis_x, angle_x = calculate_rotation_angle_axis(rotation_matrix_x_begin_for_compute, rotation_matrix_x_end_for_compute)
axis_z, angle_z = calculate_rotation_angle_axis(rotation_matrix_z_begin_for_compute, rotation_matrix_z_end_for_compute)
axis_y = np.cross(axis_z, axis_x)
axis_y = axis_y / np.linalg.norm(axis_y)
axis_z = np.cross(axis_x, axis_y)
axis_z = axis_z / np.linalg.norm(axis_z)
rotation_matrix_robot_world_to_vr_world_right = np.column_stack([-axis_z, -axis_y, -axis_x])
rotation_matrix_vr_world_to_robot_world_right = rotation_matrix_robot_world_to_vr_world_right.T
rotation_matrix_robot_world_to_vr_world_right


array([[-0.1357389 , -0.95284285,  0.27141382],
       [ 0.70530223,  0.09945726,  0.7018953 ],
       [-0.69578999,  0.28670327,  0.65854197]])

In [7]:

rotation_matrix_x_begin = np.array(
[
    [-0.9859989986278996, 0.03497319521904603, -0.163041006723636],
    [0.13307699945307244, 0.7542069970764952, -0.6430100028404236],
    [0.10047799234985519, -0.6557049967479951, -0.7483020039701881],
])
rotation_matrix_x_end =  np.array(
[
    [-0.15654899774429612, -0.5876689979017898, 0.7938120036294233],
    [-0.9480670002348389, -0.1358980038629443, -0.28757700727418595],
    [0.2768779925459307, -0.7976069952402578, -0.5358750017178925],
])

rotation_matrix_z_begin = np.array(
[
    [-0.9884239985773612, 0.02732739524532348, -0.1492380066582068],
    [0.12358999903360454, 0.7155649971963728, -0.6875260031670318],
    [0.08800129240074661, -0.6980109966069644, -0.7106590038309135],
])

rotation_matrix_z_end = np.array(
[
    [0.2527209922545782, -0.6804729959145741, -0.687814002728278],
    [-0.020046200804024826, 0.7070559965621759, -0.7068740041236252],
    [0.9673309997384207, 0.19243000367023358, 0.16504700667756203],
])



rotation_matrix_x_begin_for_compute = np.eye(3)
rotation_matrix_x_end_for_compute = rotation_matrix_x_end.T @ rotation_matrix_x_begin
rotation_matrix_z_begin_for_compute = np.eye(3)
rotation_matrix_z_end_for_compute = rotation_matrix_z_end.T @ rotation_matrix_z_begin


axis_x, angle_x = calculate_rotation_angle_axis(rotation_matrix_x_begin_for_compute, rotation_matrix_x_end_for_compute)
axis_z, angle_z = calculate_rotation_angle_axis(rotation_matrix_z_begin_for_compute, rotation_matrix_z_end_for_compute)
axis_y = np.cross(axis_z, axis_x)
axis_y = axis_y / np.linalg.norm(axis_y)
axis_z = np.cross(axis_x, axis_y)
axis_z = axis_z / np.linalg.norm(axis_z)
rotation_matrix_robot_world_to_vr_world_left = np.column_stack([-axis_z, axis_y, axis_x])
rotation_matrix_vr_world_to_robot_world_left = rotation_matrix_robot_world_to_vr_world_left.T
rotation_matrix_robot_world_to_vr_world_left


array([[ 0.03085547, -0.95049694, -0.30919817],
       [ 0.73447024, -0.1882562 ,  0.65200695],
       [-0.67793909, -0.24721484,  0.69230298]])